# YOLO26-seg Unified Multi-Domain Finetuning on `coffee_rice_v002`

Instance segmentation and disease diagnosis pipeline using **YOLO26-seg**.
Supports **Joint Multi-Domain Training (Single Unified Model for 7 Disease Classes)** to eliminate inference routing overhead, with optional domain-specific configuration (`TARGET_DOMAIN = 'joint' | 'coffee' | 'rice'`).

### Detection Taxonomy (7 Classes in Joint Mode):
- **Coffee (Classes 0..3):** `LeafMiner` (0), `PowderyMildew` (1), `Rust` (2), `AlgalLeafSpot` (3)
- **Rice (Classes 4..6):** `BrownSpot` (4), `Hispa` (5), `LeafBlast` (6)
- **Healthy:** Image-level label -> converted to shared **background frames** (empty `.txt` label file).

### Technical Specifications:
1. **Multi-Domain Joint Representation:** Coffee and Rice datasets are trained jointly in a unified feature space. The shared backbone learns general botanical leaf features while the 7-class head discriminates all disease categories simultaneously.
2. **Zero-Routing Serving Architecture:** A single ONNX runtime session (`yolo26_unified.onnx`) serves both crops directly, eliminating filename heuristics and uncalibrated cross-model confidence comparisons.
3. **Controlled Negative Ratio:** `negative_train_ratio = 0.15` in the training split prevents the 52% healthy rice frames from suppressing disease recall, while 100% of negative images are retained in validation/test for unbiased false-positive evaluation.
4. **Resolution & Augmentation:** 1024px input size preserving fine lesion structures with balanced rotation (`degrees=15.0`) and delayed mosaic closure (`close_mosaic=20`).

## 0. Dependencies

In [1]:
import importlib.util, subprocess, sys

required = {"ultralytics": "ultralytics", "pycocotools": "pycocotools", "yaml": "pyyaml"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import ultralytics
print("ultralytics", ultralytics.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics 8.4.159


## 1. Configuration

In [2]:
from __future__ import annotations

import json, os, platform, random, shutil, time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Domain switcher: 'joint' (default 7-class unified), 'coffee' (4-class), or 'rice' (3-class)
TARGET_DOMAIN = os.environ.get("TARGET_DOMAIN", "joint").lower()
assert TARGET_DOMAIN in {"joint", "unified", "both", "all", "rice", "coffee"}, f"Unsupported domain: {TARGET_DOMAIN}"
IS_JOINT = TARGET_DOMAIN in {"joint", "unified", "both", "all"}

DATASET_VERSION = os.environ.get("DATASET_VERSION", "coffee_rice_v002")
IMAGES_VERSION = os.environ.get("IMAGES_DATASET_VERSION", "coffee_rice_v001")

RUN_ID = os.environ.get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Domain-specific augmentation & training configurations
if IS_JOINT:
    TRAIN_ARGS = {
        "model": os.environ.get("YOLO_MODEL", "yolo26n-seg.pt"),
        "imgsz": int(os.environ.get("YOLO_IMGSZ", "1024")),
        "epochs": int(os.environ.get("YOLO_EPOCHS", "120")),
        "batch": int(os.environ.get("YOLO_BATCH", "8")),
        "patience": int(os.environ.get("YOLO_PATIENCE", "30")),
        "optimizer": os.environ.get("YOLO_OPTIMIZER", "AdamW"),
        "lr0": float(os.environ.get("YOLO_LR0", "1e-3")),
        "lrf": 0.01,
        "cos_lr": True,
        "warmup_epochs": 5.0,
        "weight_decay": 5e-4,
        "box": 7.5, "cls": 0.55, "dfl": 1.5,
        "mosaic": 0.9, "close_mosaic": 20, "copy_paste": 0.2, "mixup": 0.0,
        "scale": 0.5, "degrees": 15.0, "translate": 0.1, "shear": 0.0, "perspective": 0.0,
        "fliplr": 0.5, "flipud": 0.2,
        "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
        "erasing": 0.0, "overlap_mask": True, "mask_ratio": 4,
        "workers": int(os.environ.get("YOLO_WORKERS", "2")),
        "seed": SEED, "deterministic": True, "plots": True, "val": True, "save": True,
    }
elif TARGET_DOMAIN == "coffee":
    TRAIN_ARGS = {
        "model": os.environ.get("YOLO_MODEL", "yolo26n-seg.pt"),
        "imgsz": int(os.environ.get("YOLO_IMGSZ", "1024")),
        "epochs": int(os.environ.get("YOLO_EPOCHS", "120")),
        "batch": int(os.environ.get("YOLO_BATCH", "8")),
        "patience": int(os.environ.get("YOLO_PATIENCE", "30")),
        "optimizer": os.environ.get("YOLO_OPTIMIZER", "AdamW"),
        "lr0": float(os.environ.get("YOLO_LR0", "1e-3")),
        "lrf": 0.01,
        "cos_lr": True,
        "warmup_epochs": 5.0,
        "weight_decay": 5e-4,
        "box": 7.5, "cls": 0.5, "dfl": 1.5,
        "mosaic": 1.0, "close_mosaic": 15, "copy_paste": 0.3, "mixup": 0.0,
        "scale": 0.5, "degrees": 20.0, "translate": 0.1, "shear": 0.0, "perspective": 0.0,
        "fliplr": 0.5, "flipud": 0.5,
        "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
        "erasing": 0.0, "overlap_mask": True, "mask_ratio": 4,
        "workers": int(os.environ.get("YOLO_WORKERS", "2")),
        "seed": SEED, "deterministic": True, "plots": True, "val": True, "save": True,
    }
else:  # rice
    TRAIN_ARGS = {
        "model": os.environ.get("YOLO_MODEL", "yolo26n-seg.pt"),
        "imgsz": int(os.environ.get("YOLO_IMGSZ", "1024")),
        "epochs": int(os.environ.get("YOLO_EPOCHS", "120")),
        "batch": int(os.environ.get("YOLO_BATCH", "8")),
        "patience": int(os.environ.get("YOLO_PATIENCE", "30")),
        "optimizer": os.environ.get("YOLO_OPTIMIZER", "AdamW"),
        "lr0": float(os.environ.get("YOLO_LR0", "1e-3")),
        "lrf": 0.01,
        "cos_lr": True,
        "warmup_epochs": 5.0,
        "weight_decay": 5e-4,
        "box": 7.5, "cls": 0.6, "dfl": 1.5,
        "mosaic": 0.8, "close_mosaic": 25, "copy_paste": 0.15, "mixup": 0.0,
        "scale": 0.3, "degrees": 10.0, "translate": 0.1, "shear": 0.0, "perspective": 0.0,
        "fliplr": 0.5, "flipud": 0.1,
        "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
        "erasing": 0.0, "overlap_mask": True, "mask_ratio": 4,
        "workers": int(os.environ.get("YOLO_WORKERS", "2")),
        "seed": SEED, "deterministic": True, "plots": True, "val": True, "save": True,
    }

NEGATIVE_TRAIN_RATIO = float(os.environ.get("NEGATIVE_TRAIN_RATIO", "0.15"))
KEEP_ALL_NEGATIVES_IN_EVAL = True
CONF_SWEEP = np.round(np.arange(0.05, 0.91, 0.05), 2).tolist()
EVAL_MAX_IMAGES = int(os.environ.get("EVAL_MAX_IMAGES", "0")) or None

RUN_SMOKE_TEST = os.environ.get("RUN_SMOKE_TEST", "1") == "1"
RUN_FULL_TRAINING = os.environ.get("RUN_FULL_TRAINING", "1") == "1"
SMOKE_FRACTION = float(os.environ.get("SMOKE_FRACTION", "0.1"))


def find_dataset_root() -> Path:
    # 1. Check explicit environment overrides
    for key in ("DATASET_ROOT", "CLEAN_DATASET_ROOT", "PROJECT_ROOT"):
        val = os.environ.get(key)
        if val:
            for cand in [Path(val) / "data" / "clean" / DATASET_VERSION, Path(val) / DATASET_VERSION, Path(val)]:
                if (cand / "coffee").is_dir() and (cand / "rice").is_dir():
                    return cand.resolve()

    # 2. Recursive search under /kaggle/input (handles any nesting depth or slug name)
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        # First priority: look for directory having both coffee and rice with manifests
        for dirpath, dirnames, _ in os.walk(kaggle):
            dp = Path(dirpath)
            if "coffee" in dirnames and "rice" in dirnames:
                # Check for v002 markers
                if (dp / "coffee" / "manifests" / "images.csv").is_file() or (dp / "coffee" / "annotations").is_dir():
                    return dp.resolve()
                if (dp / "dataset_manifest.json").is_file() or (dp / "repair_config.json").is_file():
                    return dp.resolve()
                return dp.resolve()
        
        # Second priority: check if single domain exists under kaggle
        if not IS_JOINT:
            for dirpath, dirnames, _ in os.walk(kaggle):
                dp = Path(dirpath)
                if TARGET_DOMAIN in dirnames:
                    domain_dir = dp / TARGET_DOMAIN
                    if (domain_dir / "manifests" / "images.csv").is_file() or (domain_dir / "annotations").is_dir():
                        return dp.resolve()

    # 3. Recursive search in local workspace
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        for cand in [parent / "data" / "clean" / DATASET_VERSION, parent / DATASET_VERSION, parent]:
            if (cand / "coffee").is_dir() and (cand / "rice").is_dir():
                return cand.resolve()
    for dirpath, dirnames, _ in os.walk(here):
        dp = Path(dirpath)
        if "coffee" in dirnames and "rice" in dirnames:
            return dp.resolve()

    # Diagnostic listing if not found
    found_dirs = []
    if kaggle.is_dir():
        for dirpath, _, _ in os.walk(kaggle):
            found_dirs.append(dirpath)
    raise FileNotFoundError(
        f"Could not locate dataset root for {DATASET_VERSION}.\n"
        f"Searched all directories under /kaggle/input:\n" + "\n".join(f" - {d}" for d in found_dirs[:30])
    )


DATASET_ROOT = find_dataset_root()


def resolve_images_root(dataset_root: Path) -> Path:
    # In v002, images are self-contained inside coffee/images and rice/images
    if (dataset_root / "rice" / "images").is_dir() or (dataset_root / "coffee" / "images").is_dir():
        return dataset_root
    for d in ("rice", "coffee"):
        if (dataset_root / d).is_dir():
            sample_files = list((dataset_root / d).glob("*/*.*"))[:1]
            if sample_files:
                return dataset_root
    # Fallback to separate images dataset if mounted
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        for dirpath, dirnames, _ in os.walk(kaggle):
            dp = Path(dirpath)
            if ("rice" in dirnames or (dp / "rice" / "images").is_dir()) and ("coffee" in dirnames or (dp / "coffee" / "images").is_dir()):
                return dp.resolve()
    return dataset_root


IMAGES_ROOT = resolve_images_root(DATASET_ROOT)

WORK_ROOT = Path(os.environ.get("WORK_ROOT", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
YOLO_DATASET_DIR = WORK_ROOT / "yolo_dataset" / f"{TARGET_DOMAIN}_{DATASET_VERSION}"
RUNS_DIR = WORK_ROOT / "runs" / "yolo26_seg"
ARTIFACTS_DIR = WORK_ROOT / "artifacts" / f"yolo26_seg_{TARGET_DOMAIN}_{RUN_ID}"
for path in (RUNS_DIR, ARTIFACTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

DEFAULT_CLASSES = {
    "coffee": ["LeafMiner", "PowderyMildew", "Rust", "AlgalLeafSpot"],
    "rice": ["BrownSpot", "Hispa", "LeafBlast"],
}

if IS_JOINT:
    coffee_map = DATASET_ROOT / "coffee" / "class_mapping.json"
    rice_map = DATASET_ROOT / "rice" / "class_mapping.json"
    COFFEE_CLASSES = json.loads(coffee_map.read_text())["detection_classes"] if coffee_map.is_file() else DEFAULT_CLASSES["coffee"]
    RICE_CLASSES = json.loads(rice_map.read_text())["detection_classes"] if rice_map.is_file() else DEFAULT_CLASSES["rice"]
    CLASS_NAMES = COFFEE_CLASSES + RICE_CLASSES
else:
    DOMAIN_ROOT = DATASET_ROOT / TARGET_DOMAIN
    domain_map = DOMAIN_ROOT / "class_mapping.json"
    CLASS_NAMES = json.loads(domain_map.read_text())["detection_classes"] if domain_map.is_file() else DEFAULT_CLASSES[TARGET_DOMAIN]

print("dataset :", DATASET_ROOT)
print("images  :", IMAGES_ROOT)
print("target  :", TARGET_DOMAIN, f"({len(CLASS_NAMES)} classes: {CLASS_NAMES})")
print("device  :", DEVICE, "| run:", RUN_ID)
print("artifacts:", ARTIFACTS_DIR)


dataset : /kaggle/input/datasets/tunah72/cleaned-coffee-and-rice-leaf-disease-v002/coffee_rice_v002
images  : /kaggle/input/datasets/tunah72/cleaned-coffee-and-rice-leaf-disease-v002/coffee_rice_v002
target  : joint (7 classes: ['LeafMiner', 'PowderyMildew', 'Rust', 'AlgalLeafSpot', 'BrownSpot', 'Hispa', 'LeafBlast'])
device  : 0 | run: 20260922T115556Z
artifacts: /kaggle/working/artifacts/yolo26_seg_joint_20260922T115556Z


## 2. Load repaired manifests and re-assert the dataset invariants

In [3]:
if (DATASET_ROOT / "repair_config.json").is_file():
    REPAIR_CONFIG = json.loads((DATASET_ROOT / "repair_config.json").read_text())
elif (DATASET_ROOT / "metadata" / "preprocessing_config.json").is_file():
    REPAIR_CONFIG = json.loads((DATASET_ROOT / "metadata" / "preprocessing_config.json").read_text())
else:
    REPAIR_CONFIG = {
        "instance_policy": {"min_area_frac": 5e-4, "max_area_frac": 0.90},
        "class_policy": {"image_level_labels": ["Healthy"]},
    }

if IS_JOINT:
    manifest_rows = []
    combined_images = []
    combined_anns = []
    
    # Domain 1: Coffee (classes 0..3)
    coffee_manifest = pd.read_csv(DATASET_ROOT / "coffee" / "manifests" / "images.csv")
    coffee_manifest["domain"] = "coffee"
    coffee_manifest["sample_id"] = "coffee_" + coffee_manifest["sample_id"].astype(str)
    coffee_manifest["group_id"] = "coffee_" + coffee_manifest["group_id"].astype(str)
    with (DATASET_ROOT / "coffee" / "annotations" / "instances.coco.json").open("r", encoding="utf-8") as handle:
        coffee_coco = json.load(handle)
    
    # Invariant checks for Coffee
    crossing_c = coffee_manifest.groupby("group_id")["split"].nunique()
    assert int((crossing_c > 1).sum()) == 0, "coffee group spans multiple splits"
    assert int((coffee_manifest.groupby("md5")["split"].nunique() > 1).sum()) == 0, "coffee md5 across splits"
    
    # Domain 2: Rice (classes 4..6, image_id offset 1_000_000)
    rice_manifest = pd.read_csv(DATASET_ROOT / "rice" / "manifests" / "images.csv")
    rice_manifest["domain"] = "rice"
    rice_manifest["sample_id"] = "rice_" + rice_manifest["sample_id"].astype(str)
    rice_manifest["group_id"] = "rice_" + rice_manifest["group_id"].astype(str)
    rice_manifest["coco_image_id"] = rice_manifest["coco_image_id"].astype(int) + 1_000_000
    with (DATASET_ROOT / "rice" / "annotations" / "instances.coco.json").open("r", encoding="utf-8") as handle:
        rice_coco = json.load(handle)
        
    # Invariant checks for Rice
    crossing_r = rice_manifest.groupby("group_id")["split"].nunique()
    assert int((crossing_r > 1).sum()) == 0, "rice group spans multiple splits"
    assert int((rice_manifest.groupby("md5")["split"].nunique() > 1).sum()) == 0, "rice md5 across splits"
    
    MANIFEST = pd.concat([coffee_manifest, rice_manifest], ignore_index=True)
    
    # Merge COCO
    categories = [{"id": i, "name": name} for i, name in enumerate(CLASS_NAMES)]
    for img in coffee_coco["images"]:
        img_copy = dict(img)
        img_copy["domain"] = "coffee"
        combined_images.append(img_copy)
    for ann in coffee_coco["annotations"]:
        combined_anns.append(dict(ann)) # category_id 0..3 unchanged
        
    for img in rice_coco["images"]:
        img_copy = dict(img)
        img_copy["id"] = int(img["id"]) + 1_000_000
        img_copy["domain"] = "rice"
        combined_images.append(img_copy)
    for ann in rice_coco["annotations"]:
        ann_copy = dict(ann)
        ann_copy["id"] = int(ann["id"]) + 1_000_000
        ann_copy["image_id"] = int(ann["image_id"]) + 1_000_000
        ann_copy["category_id"] = int(ann["category_id"]) + len(COFFEE_CLASSES) # offset 4..6
        combined_anns.append(ann_copy)
        
    COCO = {
        "info": {"description": "Joint Coffee & Rice Leaf Disease v002"},
        "categories": categories,
        "images": combined_images,
        "annotations": combined_anns
    }
else:
    DOMAIN_ROOT = DATASET_ROOT / TARGET_DOMAIN
    MANIFEST = pd.read_csv(DOMAIN_ROOT / "manifests" / "images.csv")
    MANIFEST["domain"] = TARGET_DOMAIN
    with (DOMAIN_ROOT / "annotations" / "instances.coco.json").open("r", encoding="utf-8") as handle:
        COCO = json.load(handle)
    crossing = MANIFEST.groupby("group_id")["split"].nunique()
    assert int((crossing > 1).sum()) == 0, "group spans multiple splits"
    assert int((MANIFEST.groupby("md5")["split"].nunique() > 1).sum()) == 0, "md5 across splits"

assert [c["name"] for c in sorted(COCO["categories"], key=lambda c: c["id"])] == CLASS_NAMES
assert set(MANIFEST["split"]) <= {"train", "val", "test"}

# Global invariants: single ring, area fraction bounds, background label sanity
areas = []
for ann in COCO["annotations"]:
    assert len(ann["segmentation"]) == 1, "annotation has more than one ring"
    assert len(ann["segmentation"][0]) >= 6, "ring has fewer than 3 points"
    areas.append(ann["area"])

sizes = {int(img["id"]): img["width"] * img["height"] for img in COCO["images"]}
fracs = np.array([ann["area"] / sizes[int(ann["image_id"])] for ann in COCO["annotations"]])
assert fracs.min() >= REPAIR_CONFIG["instance_policy"]["min_area_frac"]
assert fracs.max() <= REPAIR_CONFIG["instance_policy"]["max_area_frac"]

negatives = MANIFEST[MANIFEST["is_negative"] == 1]
assert set(negatives["image_label"]) <= set(REPAIR_CONFIG["class_policy"]["image_level_labels"]),     "a diseased image is marked as background"

print(f"Total images={len(MANIFEST)} instances={len(COCO['annotations'])} "
      f"background={len(negatives)} groups={MANIFEST['group_id'].nunique()}")
print(pd.crosstab(MANIFEST["image_label"], MANIFEST["split"]).to_string())
print("instance area fraction: p05={:.4f} median={:.4f} p95={:.4f}".format(
    *np.percentile(fracs, [5, 50, 95])))


Total images=4324 instances=3107 background=1472 groups=1916
split          test  train  val
image_label                    
AlgalLeafSpot    46    216   47
BrownSpot        71    319   68
Healthy         208   1036  228
Hispa            90    379   68
LeafBlast        92    414   95
LeafMiner        59    277   60
PowderyMildew    13     61   13
Rust             69    325   70
instance area fraction: p05=0.0074 median=0.1424 p95=0.3222


## 3. Export the YOLO-seg dataset

Rules that differ from the v001 exporter:

1. **one label line per annotation** (the repaired ring), never one per COCO ring;
2. background images get an **empty** `.txt` so Ultralytics treats them as negatives;
3. the background share of the training split is capped at `NEGATIVE_TRAIN_RATIO`
   (val/test keep every background image so false positives stay measurable);
4. images are symlinked when possible, so a 8 GB dataset is not duplicated.

In [4]:
def yolo_polygon(ring: list[float], width: int, height: int) -> list[float] | None:
    xs = np.clip(np.asarray(ring[0::2], dtype=np.float64) / width, 0.0, 1.0)
    ys = np.clip(np.asarray(ring[1::2], dtype=np.float64) / height, 0.0, 1.0)
    if xs.size < 3:
        return None
    return np.stack([xs, ys], axis=1).ravel().tolist()


def select_training_negatives(manifest: pd.DataFrame, ratio: float) -> pd.DataFrame:
    out = manifest.copy()
    out["used"] = True
    train = out[out["split"] == "train"]
    positives = int((train["is_negative"] == 0).sum())
    budget = int(round(positives * ratio / max(1e-9, 1.0 - ratio)))
    negatives = train[train["is_negative"] == 1]
    if len(negatives) > budget:
        keep = negatives.sample(n=budget, random_state=SEED)["sample_id"]
        drop = set(negatives["sample_id"]) - set(keep)
        out.loc[out["sample_id"].isin(drop), "used"] = False
    print(f"train positives={positives} background_available={len(negatives)} "
          f"background_used={min(len(negatives), budget)}")
    return out


def export_yolo_dataset(manifest: pd.DataFrame) -> pd.DataFrame:
    anns_by_image = defaultdict(list)
    for ann in COCO["annotations"]:
        anns_by_image[int(ann["image_id"])].append(ann)
    if YOLO_DATASET_DIR.exists():
        shutil.rmtree(YOLO_DATASET_DIR)

    rows = []
    for row in manifest[manifest["used"]].itertuples():
        split = row.split
        image_dir = YOLO_DATASET_DIR / "images" / split
        label_dir = YOLO_DATASET_DIR / "labels" / split
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)

        domain = getattr(row, "domain", TARGET_DOMAIN)
        norm_name = str(row.coco_file_name).replace("\\", "/")
        candidates = [
            IMAGES_ROOT / domain / norm_name,
            IMAGES_ROOT / norm_name,
            DATASET_ROOT / domain / norm_name,
            IMAGES_ROOT / domain / "images" / Path(norm_name).name,
            DATASET_ROOT / domain / "images" / Path(norm_name).name,
            IMAGES_ROOT / domain / Path(norm_name).name,
            DATASET_ROOT / domain / Path(norm_name).name,
        ]
        source = None
        for c in candidates:
            if c.is_file():
                source = c.resolve()
                break
        if source is None:
            raise FileNotFoundError(f"Image not found for {domain}/{row.coco_file_name}. Tried: {[str(c) for c in candidates]}")
        target = image_dir / f"{row.sample_id}{source.suffix}"
        if not target.exists():
            try:
                target.symlink_to(source)
            except OSError:
                shutil.copy2(source, target)

        lines = []
        for ann in anns_by_image.get(int(row.coco_image_id), []):
            polygon = yolo_polygon(ann["segmentation"][0], int(row.width), int(row.height))
            if polygon is None:
                continue
            coords = " ".join(f"{v:.6f}" for v in polygon)
            lines.append(f"{int(ann['category_id'])} {coords}")
        label_path = label_dir / f"{row.sample_id}.txt"
        label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

        rows.append({"sample_id": row.sample_id, "split": split, "image_label": row.image_label,
                     "image_path": str(target), "label_path": str(label_path), "domain": domain,
                     "num_instances": len(lines), "width": int(row.width), "height": int(row.height),
                     "coco_image_id": int(row.coco_image_id), "group_id": row.group_id})
    return pd.DataFrame(rows)


MANIFEST = select_training_negatives(MANIFEST, NEGATIVE_TRAIN_RATIO)
EXPORT = export_yolo_dataset(MANIFEST)

DATA_YAML = YOLO_DATASET_DIR / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(YOLO_DATASET_DIR.resolve()),
    "train": "images/train", "val": "images/val", "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    "nc": len(CLASS_NAMES),
}, sort_keys=False), encoding="utf-8")

print(EXPORT.groupby("split").agg(images=("sample_id", "size"),
                                  instances=("num_instances", "sum"),
                                  background=("num_instances", lambda s: int((s == 0).sum()))).to_string())
print(DATA_YAML.read_text())


train positives=1991 background_available=1036 background_used=351
       images  instances  background
split                               
test      648        505         208
train    2342       2147         351
val       649        455         228
path: /kaggle/working/yolo_dataset/joint_coffee_rice_v002
train: images/train
val: images/val
test: images/test
names:
  0: LeafMiner
  1: PowderyMildew
  2: Rust
  3: AlgalLeafSpot
  4: BrownSpot
  5: Hispa
  6: LeafBlast
nc: 7



## 4. Label QA on the exported dataset

In [5]:
def read_label(path: Path) -> list[tuple[int, np.ndarray]]:
    out = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if not parts:
            continue
        out.append((int(parts[0]), np.asarray(parts[1:], dtype=np.float64).reshape(-1, 2)))
    return out


exported_instances = 0
for row in EXPORT.itertuples():
    for class_id, coords in read_label(row.label_path):
        assert 0 <= class_id < len(CLASS_NAMES), f"bad class in {row.label_path}"
        assert coords.shape[0] >= 3, f"ring < 3 points in {row.label_path}"
        assert coords.min() >= 0.0 and coords.max() <= 1.0, f"out-of-range ring in {row.label_path}"
        exported_instances += 1
expected = sum(1 for ann in COCO["annotations"]
               if int(ann["image_id"]) in set(EXPORT["coco_image_id"]))
assert exported_instances == expected, f"instance count mismatch: {exported_instances} != {expected}"
print(f"label QA passed: {exported_instances} instances, one line per repaired annotation")

per_class = defaultdict(int)
for row in EXPORT.itertuples():
    for class_id, _ in read_label(row.label_path):
        per_class[CLASS_NAMES[class_id]] += 1
label_stats = pd.DataFrame(sorted(per_class.items()), columns=["class", "instances"])
label_stats.to_csv(ARTIFACTS_DIR / "label_distribution.csv", index=False)
display(label_stats)


def overlay(row, out_path: Path) -> None:
    image = Image.open(row.image_path).convert("RGB")
    draw = ImageDraw.Draw(image, "RGBA")
    palette = ["#E7298A", "#1B9E77", "#7570B3", "#D95F02", "#E6AB02", "#A6761D", "#66A61E"]
    for class_id, coords in read_label(row.label_path):
        points = [(float(x) * image.width, float(y) * image.height) for x, y in coords]
        draw.polygon(points, fill=palette[class_id % len(palette)] + "66",
                     outline=palette[class_id % len(palette)], width=4)
    image.thumbnail((640, 640))
    image.save(out_path)


qa_dir = ARTIFACTS_DIR / "label_qa"; qa_dir.mkdir(exist_ok=True)
sample = EXPORT[EXPORT["num_instances"] > 0].sample(min(8, int((EXPORT["num_instances"] > 0).sum())),
                                                    random_state=SEED)
for row in sample.itertuples():
    overlay(row, qa_dir / f"{row.sample_id}.jpg")
print("wrote label overlays:", sorted(p.name for p in qa_dir.iterdir()))


label QA passed: 3107 instances, one line per repaired annotation


,class,instances
0,AlgalLeafSpot,309
1,BrownSpot,538
2,Hispa,631
3,LeafBlast,682
4,LeafMiner,396
5,PowderyMildew,87
6,Rust,464


wrote label overlays: ['coffee_coffee_0003961.jpg', 'coffee_coffee_0004667.jpg', 'coffee_coffee_0006450.jpg', 'rice_rice_0000096.jpg', 'rice_rice_0000394.jpg', 'rice_rice_0002468.jpg', 'rice_rice_0002510.jpg', 'rice_rice_0002855.jpg']


## 5. Train

`RUN_SMOKE_TEST=1` runs one bounded epoch on a fraction of the data to prove the pipeline before
committing GPU hours. Every training argument, the resolved environment, and the Ultralytics
`results.csv` are saved as artifacts so the run can be audited later.

In [6]:
from ultralytics import YOLO

COMMON = {k: v for k, v in TRAIN_ARGS.items() if k != "model"}
COMMON |= {"data": str(DATA_YAML), "project": str(RUNS_DIR), "device": DEVICE, "exist_ok": True}

if RUN_SMOKE_TEST:
    started = time.time()
    smoke = YOLO(TRAIN_ARGS["model"])
    smoke.train(**{**COMMON, "name": f"smoke_{TARGET_DOMAIN}_{RUN_ID}", "epochs": 1,
                   "fraction": SMOKE_FRACTION, "patience": 1, "close_mosaic": 0, "plots": False})
    print(f"smoke test ok in {time.time() - started:.0f}s")
else:
    print("smoke test skipped")

Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.55, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo_dataset/joint_coffee_rice_v002/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=0.1, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=0.9, multi_scale=0.0, name=smoke_joint

libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 39/41 3.6it/s 37.3s<0.6s

libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 1.0s/it 41.7s
                   all        649        455    0.00478      0.319    0.00162   0.000412    0.00566      0.384     0.0058    0.00153

1 epochs completed in 0.019 hours.
Optimizer stripped from /kaggle/working/runs/yolo26_seg/smoke_joint_20260922T115556Z/weights/last.pt, 6.6MB


/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


Optimizer stripped from /kaggle/working/runs/yolo26_seg/smoke_joint_20260922T115556Z/weights/best.pt, 6.6MB

Validating /kaggle/working/runs/yolo26_seg/smoke_joint_20260922T115556Z/weights/best.pt...
Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-seg summary (fused): 136 layers, 2,690,249 parameters, 0 gradients, 9.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 39/41 4.2it/s 29.9s<0.5s

libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 1.2it/s 33.4s
                   all        649        455    0.00474       0.32    0.00164   0.000424     0.0065      0.386    0.00607    0.00164
             LeafMiner         60         60          0          0          0          0          0          0          0          0
         PowderyMildew         13         13   0.000391          1   0.000486   0.000218   0.000391          1   0.000482   0.000297
                  Rust         70         70      0.019     0.0429   0.000957   0.000577      0.019     0.0429   0.000957   0.000676
         AlgalLeafSpot         47         47    0.00957      0.766    0.00915      0.002    0.00718      0.574    0.00529    0.00167
             BrownSpot         68         77    0.00348       0.13   0.000745   0.000144     0.0111      0.416     0.0277    0.00726
                 Hispa        

In [7]:
TRAIN_RUN_NAME = f"yolo26n_seg_{TARGET_DOMAIN}_{RUN_ID}"
train_summary = {"executed": False}

if RUN_FULL_TRAINING:
    started = time.time()
    model = YOLO(TRAIN_ARGS["model"])
    results = model.train(**{**COMMON, "name": TRAIN_RUN_NAME})
    train_dir = Path(results.save_dir)
    best_ckpt = train_dir / "weights" / "best.pt"
    train_summary = {
        "executed": True,
        "run_dir": str(train_dir),
        "best_checkpoint": str(best_ckpt),
        "wall_time_seconds": round(time.time() - started, 1),
        "epochs_requested": TRAIN_ARGS["epochs"],
    }
    curves = pd.read_csv(train_dir / "results.csv")
    train_summary["epochs_completed"] = int(curves["epoch"].max())
    curves.to_csv(ARTIFACTS_DIR / "training_curves.csv", index=False)
    for name in ("results.png", "confusion_matrix_normalized.png", "MaskPR_curve.png", "BoxPR_curve.png"):
        source = train_dir / name
        if source.exists():
            shutil.copy2(source, ARTIFACTS_DIR / name)
    display(curves.tail(5))
else:
    candidates = sorted(RUNS_DIR.glob(f"yolo26n_seg_{TARGET_DOMAIN}*/weights/best.pt"),
                        key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise RuntimeError("No checkpoint available and RUN_FULL_TRAINING=0")
    best_ckpt = candidates[-1]
    train_summary = {"executed": False, "best_checkpoint": str(best_ckpt), "reused": True}

print(json.dumps(train_summary, indent=2))

Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=20, cls=0.55, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo_dataset/joint_coffee_rice_v002/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=0.9, multi_scale=0.0, name=yolo26n_

libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 39/41 4.0it/s 20.3s<0.5s

libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 2.0it/s 20.7s
                   all        649        455      0.204      0.349      0.157     0.0625      0.116      0.169     0.0605     0.0148

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      2/120      5.06G      0.953      0.846      2.227     0.0193      1.877         12       1024: 100% ━━━━━━━━━━━━ 293/293 2.5it/s 1:55
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 39/41 4.1it/s 20.1s<0.5s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 2.0it/s 20.6s
                   all        649        455      0.829      0.774      0.817      0.755      0.829      0.774      0.816      0.791

120 epochs completed in 4.526 hours.
Optimizer stripped from /kaggle/working/runs/yolo26_seg/yolo26n_seg_joint_20260922T115556Z/weights/last.pt, 6.6MB
Optimizer stripped from /kaggle/working/runs/yolo26_seg/yolo26n_seg_joint_20260922T115556Z/weights/best.pt, 6.6MB

Validating /kaggle/working/runs/yolo26_seg/yolo26n_seg_joint_20260922T115556Z/weights/best.pt...
Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-seg summary (fused): 136 layers, 2,690,249 parameters, 0 gradients, 9.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 97% ━━━━━━━━━━━╸ 40/41 4.7

libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 2.1it/s 19.1s
                   all        649        455      0.832      0.769      0.819      0.761      0.832      0.769      0.818      0.798
             LeafMiner         60         60       0.91      0.848      0.867      0.778       0.91      0.848      0.867      0.819
         PowderyMildew         13         13      0.968      0.846      0.972      0.916      0.968      0.846      0.972      0.972
                  Rust         70         70      0.937      0.854      0.886      0.837      0.937      0.854      0.886      0.881
         AlgalLeafSpot         47         47       0.92      0.978      0.978      0.921       0.92      0.978      0.978      0.978
             BrownSpot         68         77      0.675      0.593      0.705      0.637      0.675      0.593      0.706      0.657
                 Hispa        

,epoch,time,train/box_loss,train/seg_loss,train/cls_loss,train/l1_loss,train/sem_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),...,metrics/mAP50(M),metrics/mAP50-95(M),val/box_loss,val/seg_loss,val/cls_loss,val/l1_loss,val/sem_loss,lr/pg0,lr/pg1,lr/pg2
115,116,15819.1,0.31248,0.23443,0.44222,0.00819,0.15824,0.82660,0.77188,0.81820,...,0.81721,0.79194,0.43940,0.42738,1.84729,0.01333,0,0.000014,0.000014,0.000014
116,117,15937.1,0.31714,0.25409,0.47943,0.00826,0.17670,0.83847,0.76341,0.81783,...,0.81692,0.79218,0.44103,0.42594,1.88270,0.01346,0,0.000013,0.000013,0.000013
117,118,16057.4,0.31135,0.25063,0.46438,0.00812,0.16868,0.83131,0.76274,0.81744,...,0.81651,0.79260,0.42974,0.42648,1.88808,0.01286,0,0.000012,0.000012,0.000012
118,119,16175.2,0.30602,0.23045,0.45107,0.00799,0.16422,0.83628,0.76281,0.81781,...,0.81668,0.79403,0.42835,0.42723,1.88049,0.01286,0,0.000011,0.000011,0.000011
119,120,16294.6,0.31034,0.23528,0.45354,0.00807,0.16987,0.82879,0.77448,0.81728,...,0.81637,0.79140,0.43005,0.42526,1.87444,0.01293,0,0.000010,0.000010,0.000010


{
  "executed": true,
  "run_dir": "/kaggle/working/runs/yolo26_seg/yolo26n_seg_joint_20260922T115556Z",
  "best_checkpoint": "/kaggle/working/runs/yolo26_seg/yolo26n_seg_joint_20260922T115556Z/weights/best.pt",
  "wall_time_seconds": 16350.4,
  "epochs_requested": 120,
  "epochs_completed": 120
}


## 6. Ultralytics validation on val and test

In [8]:
def ultralytics_metrics(metrics) -> dict:
    def value(path):
        node = metrics
        for part in path.split("."):
            node = getattr(node, part, None)
            if node is None:
                return None
        try:
            return float(node)
        except (TypeError, ValueError):
            return None

    out = {
        "mask_mAP50": value("seg.map50"), "mask_mAP50_95": value("seg.map"),
        "mask_precision": value("seg.mp"), "mask_recall": value("seg.mr"),
        "box_mAP50": value("box.map50"), "box_mAP50_95": value("box.map"),
        "box_precision": value("box.mp"), "box_recall": value("box.mr"),
        "fitness": getattr(metrics, "fitness", None),
    }
    per_class = {}
    try:
        for index, class_id in enumerate(metrics.ap_class_index):
            per_class[CLASS_NAMES[int(class_id)]] = {
                "mask_AP50": float(metrics.seg.ap50[index]),
                "mask_AP50_95": float(metrics.seg.ap[index]),
                "box_AP50": float(metrics.box.ap50[index]),
            }
    except Exception as error:                                    # noqa: BLE001
        per_class = {"error": str(error)}
    out["per_class"] = per_class
    return out


VALIDATION = {}
for split in ("val", "test"):
    metrics = YOLO(str(best_ckpt)).val(data=str(DATA_YAML), split=split, imgsz=TRAIN_ARGS["imgsz"],
                                       device=DEVICE, plots=(split == "test"), seed=SEED,
                                       project=str(RUNS_DIR), name=f"val_{split}_{RUN_ID}",
                                       exist_ok=True)
    VALIDATION[split] = ultralytics_metrics(metrics)
    print(f"=== {split}")
    print(json.dumps({k: v for k, v in VALIDATION[split].items() if k != "per_class"}, indent=2))
    display(pd.DataFrame(VALIDATION[split]["per_class"]).T)

Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-seg summary (fused): 136 layers, 2,690,249 parameters, 0 gradients, 9.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 406.5±476.1 MB/s, size: 698.3 KB)
val: Scanning /kaggle/working/yolo_dataset/joint_coffee_rice_v002/labels/val.cache... 649 images, 228 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 649/649 226.8Mit/s 0.0s


libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 39/41 3.9it/s 20.3s<0.5s

libpng warning: iCCP: unexpected zlib return code


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 2.0it/s 20.8s
                   all        649        455      0.832      0.769      0.819      0.762      0.832      0.769      0.818      0.798
             LeafMiner         60         60       0.91      0.848      0.867      0.775       0.91      0.848      0.867      0.819
         PowderyMildew         13         13      0.967      0.846      0.972      0.916      0.967      0.846      0.972      0.972
                  Rust         70         70      0.937      0.854      0.886       0.84      0.937      0.854      0.886      0.881
         AlgalLeafSpot         47         47       0.92      0.978      0.978      0.928       0.92      0.978      0.978      0.978
             BrownSpot         68         77      0.675      0.594      0.705      0.641      0.675      0.594      0.706      0.657
                 Hispa        

,mask_AP50,mask_AP50_95,box_AP50
LeafMiner,0.867400,0.818745,0.867090
PowderyMildew,0.972376,0.972376,0.972376
Rust,0.885916,0.880653,0.885866
AlgalLeafSpot,0.978461,0.978461,0.978461
BrownSpot,0.706399,0.656927,0.704967
Hispa,0.534305,0.525253,0.542684
LeafBlast,0.779200,0.753752,0.781242


Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-seg summary (fused): 136 layers, 2,690,249 parameters, 0 gradients, 9.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 31.4±15.3 MB/s, size: 1792.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/yolo_dataset/joint_coffee_rice_v002/labels/test... 648 images, 208 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 648/648 85.8it/s 7.6s
val: New cache created: /kaggle/working/yolo_dataset/joint_coffee_rice_v002/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 1.4it/s 29.9s
                   all        648        505      0.822      0.726      0.779      0.731      0.822      0.726      0.775      0.756
             LeafMiner         

,mask_AP50,mask_AP50_95,box_AP50
LeafMiner,0.757024,0.738754,0.766770
PowderyMildew,0.925000,0.925000,0.925000
Rust,0.898931,0.881300,0.898931
AlgalLeafSpot,0.962321,0.960510,0.962321
BrownSpot,0.621948,0.572186,0.625974
Hispa,0.571364,0.556584,0.582289
LeafBlast,0.690314,0.658291,0.694527


## 7. Confidence threshold selected on the validation split

The previous run reported recall at the default `conf=0.25`, which is meaningless for a model whose
logits sit at 0.12-0.18. The operating point is selected here on **val** (never on test) by mask-F1
and then applied unchanged to test.

In [9]:
from pycocotools import mask as mask_utils


def predict_split(split: str, conf: float, iou: float = 0.7, limit: int | None = None):
    frame = EXPORT[EXPORT["split"] == split]
    if limit:
        frame = frame.head(limit)
    model = YOLO(str(best_ckpt))
    for row in frame.itertuples():
        result = model.predict(row.image_path, imgsz=TRAIN_ARGS["imgsz"], conf=conf, iou=iou,
                               device=DEVICE, retina_masks=True, verbose=False)[0]
        yield row, result


def gt_instance_masks(row) -> list[tuple[int, np.ndarray]]:
    out = []
    for class_id, coords in read_label(row.label_path):
        canvas = Image.new("L", (row.width, row.height), 0)
        ImageDraw.Draw(canvas).polygon(
            [(float(x) * row.width, float(y) * row.height) for x, y in coords], outline=1, fill=1)
        out.append((class_id, np.asarray(canvas, dtype=bool)))
    return out


def pred_instance_masks(row, result) -> list[tuple[int, float, np.ndarray]]:
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return []
    masks = result.masks.data.detach().cpu().numpy() > 0.5
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    scores = result.boxes.conf.detach().cpu().numpy()
    out = []
    for class_id, score, mask in zip(classes, scores, masks):
        if mask.shape != (row.height, row.width):
            resized = Image.fromarray(mask.astype(np.uint8) * 255).resize(
                (row.width, row.height), Image.Resampling.NEAREST)
            mask = np.asarray(resized) > 0
        out.append((int(class_id), float(score), mask))
    return out


def match_counts(gt, pred, iou_threshold: float = 0.5) -> tuple[int, int, int]:
    used = set()
    tp = 0
    for class_id, _, pred_mask in sorted(pred, key=lambda item: -item[1]):
        best_iou, best_index = 0.0, -1
        for index, (gt_class, gt_mask) in enumerate(gt):
            if index in used or gt_class != class_id:
                continue
            union = np.logical_or(pred_mask, gt_mask).sum()
            if not union:
                continue
            iou = float(np.logical_and(pred_mask, gt_mask).sum()) / float(union)
            if iou > best_iou:
                best_iou, best_index = iou, index
        if best_iou >= iou_threshold:
            used.add(best_index); tp += 1
    return tp, len(pred) - tp, len(gt) - tp


sweep_rows = []
cache = {}
for row, result in predict_split("val", conf=0.01, limit=EVAL_MAX_IMAGES):
    cache[row.sample_id] = (row, gt_instance_masks(row), pred_instance_masks(row, result))

for conf in CONF_SWEEP:
    tp = fp = fn = 0
    empty_on_background = 0
    background_total = 0
    for row, gt, pred in cache.values():
        filtered = [item for item in pred if item[1] >= conf]
        a, b, c = match_counts(gt, filtered)
        tp += a; fp += b; fn += c
        if not gt:
            background_total += 1
            empty_on_background += int(not filtered)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    sweep_rows.append({
        "conf": conf, "tp": tp, "fp": fp, "fn": fn,
        "precision": round(precision, 4), "recall": round(recall, 4),
        "f1": round(2 * precision * recall / max(1e-9, precision + recall), 4),
        "background_images": background_total,
        "background_clean_rate": round(empty_on_background / max(1, background_total), 4),
    })

SWEEP = pd.DataFrame(sweep_rows)
SWEEP.to_csv(ARTIFACTS_DIR / "val_confidence_sweep.csv", index=False)
FALLBACK_CONF = 0.25
if float(SWEEP["f1"].max()) <= 0.0:
    BEST_CONF = FALLBACK_CONF
    CONF_SELECTION = "fallback: no true positive at any threshold on val"
else:
    BEST_CONF = float(SWEEP.loc[SWEEP["f1"].idxmax(), "conf"])
    CONF_SELECTION = "val_mask_f1"
display(SWEEP)
print("operating confidence:", BEST_CONF, "|", CONF_SELECTION)
if CONF_SELECTION.startswith("fallback"):
    print("WARNING: the checkpoint matched no ground-truth instance on val. "
          "Do not publish these metrics; investigate training before evaluating.")

libpng warning: iCCP: unexpected zlib return code


,conf,tp,fp,fn,precision,recall,f1,background_images,background_clean_rate
0,0.05,366,912,89,0.2864,0.8044,0.4224,228,0.0000
1,0.10,360,397,95,0.4756,0.7912,0.5941,228,0.0351
2,0.15,356,293,99,0.5485,0.7824,0.6449,228,0.2325
3,0.20,354,229,101,0.6072,0.7780,0.6821,228,0.4254
4,0.25,349,189,106,0.6487,0.7670,0.7029,228,0.5263
5,0.30,345,158,110,0.6859,0.7582,0.7203,228,0.6053
6,0.35,341,138,114,0.7119,0.7495,0.7302,228,0.6447
7,0.40,335,123,120,0.7314,0.7363,0.7338,228,0.6886
8,0.45,328,107,127,0.7540,0.7209,0.7371,228,0.7281
9,0.50,318,87,137,0.7852,0.6989,0.7395,228,0.7807


operating confidence: 0.5 | val_mask_f1


## 8. Independent COCO evaluation at original resolution

Ultralytics validates in letterboxed space. This block scores predictions with `pycocotools`
against the repaired COCO file in the **original image coordinate system**, for both `segm` and
`bbox`, so the numbers are comparable with Mask2Former / RF-DETR once those are retrained under the
same protocol.

In [10]:
def coco_eval_on_test(conf: float, limit: int | None = None, domain_filter: str | None = None) -> dict:
    from pycocotools.coco import COCO as PyCOCO
    from pycocotools.cocoeval import COCOeval

    frame = EXPORT[EXPORT["split"] == "test"]
    if domain_filter:
        frame = frame[frame["domain"] == domain_filter]
    if limit:
        frame = frame.head(limit)
    keep_ids = set(frame["coco_image_id"])
    
    # Filter categories if domain specified
    if domain_filter == "coffee" and IS_JOINT:
        valid_cat_ids = set(range(len(COFFEE_CLASSES)))
    elif domain_filter == "rice" and IS_JOINT:
        valid_cat_ids = set(range(len(COFFEE_CLASSES), len(CLASS_NAMES)))
    else:
        valid_cat_ids = set(range(len(CLASS_NAMES)))

    subset = {
        "info": COCO.get("info", {}), "licenses": [],
        "categories": [c for c in COCO["categories"] if c["id"] in valid_cat_ids],
        "images": [img for img in COCO["images"] if int(img["id"]) in keep_ids],
        "annotations": [dict(ann) for ann in COCO["annotations"]
                        if int(ann["image_id"]) in keep_ids and ann["category_id"] in valid_cat_ids],
    }
    suffix = f"_{domain_filter}" if domain_filter else ""
    gt_path = ARTIFACTS_DIR / f"test_ground_truth{suffix}.coco.json"
    gt_path.write_text(json.dumps(subset), encoding="utf-8")

    detections, per_image = [], []
    latencies = []
    model = YOLO(str(best_ckpt))
    for row in frame.itertuples():
        started = time.perf_counter()
        result = model.predict(row.image_path, imgsz=TRAIN_ARGS["imgsz"], conf=conf, iou=0.7,
                               device=DEVICE, retina_masks=True, verbose=False)[0]
        latencies.append((time.perf_counter() - started) * 1000.0)
        predictions = pred_instance_masks(row, result)
        for class_id, score, mask in predictions:
            if class_id not in valid_cat_ids:
                continue
            rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
            rle["counts"] = rle["counts"].decode("ascii")
            ys, xs = np.where(mask)
            if not len(xs):
                continue
            detections.append({
                "image_id": int(row.coco_image_id), "category_id": int(class_id),
                "score": float(score), "segmentation": rle,
                "bbox": [float(xs.min()), float(ys.min()),
                         float(xs.max() - xs.min() + 1), float(ys.max() - ys.min() + 1)],
            })
        per_image.append({"sample_id": row.sample_id, "image_label": row.image_label,
                          "n_gt": row.num_instances, "n_pred": len(predictions),
                          "top_score": max([p[1] for p in predictions], default=0.0),
                          "pred_classes": ";".join(CLASS_NAMES[p[0]] for p in predictions)})

    predictions_path = ARTIFACTS_DIR / f"test_predictions{suffix}.coco.json"
    predictions_path.write_text(json.dumps(detections), encoding="utf-8")
    if not domain_filter:
        pd.DataFrame(per_image).to_csv(ARTIFACTS_DIR / "test_per_image_predictions.csv", index=False)

    out = {"conf": conf, "n_images": len(frame), "n_detections": len(detections),
           "domain": domain_filter or "all",
           "latency_ms_mean": round(float(np.mean(latencies)), 2) if latencies else 0.0,
           "latency_ms_p95": round(float(np.percentile(latencies, 95)), 2) if latencies else 0.0,
           "device": str(DEVICE)}
    if not detections:
        out["warning"] = "no detections above threshold"
        return out

    gt = PyCOCO(str(gt_path))
    dt = gt.loadRes(str(predictions_path))
    for iou_type in ("segm", "bbox"):
        evaluator = COCOeval(gt, dt, iou_type)
        evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
        prefix = "mask" if iou_type == "segm" else "box"
        out[f"{prefix}_mAP50_95"] = round(float(evaluator.stats[0]), 4)
        out[f"{prefix}_mAP50"] = round(float(evaluator.stats[1]), 4)
        out[f"{prefix}_mAP75"] = round(float(evaluator.stats[2]), 4)
        out[f"{prefix}_AR100"] = round(float(evaluator.stats[8]), 4)
        per_class = {}
        precisions = evaluator.eval["precision"]
        for index, category in enumerate(sorted(subset_c["id"] for subset_c in subset["categories"])):
            values = precisions[0, :, index, 0, 2]
            values = values[values > -1]
            per_class[CLASS_NAMES[category]] = round(float(values.mean()) if values.size else float("nan"), 4)
        out[f"{prefix}_AP50_per_class"] = per_class
    return out


COCO_METRICS = coco_eval_on_test(BEST_CONF, limit=EVAL_MAX_IMAGES)
print("=== Overall COCO Metrics:")
print(json.dumps(COCO_METRICS, indent=2))

DOMAIN_METRICS = {}
if IS_JOINT:
    for dom in ("coffee", "rice"):
        print(f"=== Domain Breakdown: {dom.upper()}")
        DOMAIN_METRICS[dom] = coco_eval_on_test(BEST_CONF, limit=EVAL_MAX_IMAGES, domain_filter=dom)
        print(json.dumps(DOMAIN_METRICS[dom], indent=2))
    (ARTIFACTS_DIR / "domain_breakdown_metrics.json").write_text(json.dumps(DOMAIN_METRICS, indent=2), encoding="utf-8")


loading annotations into memory...
Done (t=0.10s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.33s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.611
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.617
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.617
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.654
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.646
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.646
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

## 9. Semantic overlap, background false positives, and CPU latency

`background_clean_rate` is the share of no-disease images on which the model correctly returns
nothing. It is the metric the v001 model failed silently, and the one the serving rejection gate
depends on.

In [11]:
def semantic_scores(conf: float, split: str = "test", limit: int | None = None) -> tuple[dict, pd.DataFrame]:
    rows = []
    intersection = np.zeros(len(CLASS_NAMES)); union = np.zeros(len(CLASS_NAMES))
    dice_num = np.zeros(len(CLASS_NAMES)); dice_den = np.zeros(len(CLASS_NAMES))
    background_total = background_clean = 0
    for row, result in predict_split(split, conf=conf, limit=limit):
        gt = gt_instance_masks(row)
        pred = pred_instance_masks(row, result)
        gt_union = np.zeros((len(CLASS_NAMES), row.height, row.width), dtype=bool)
        pred_union = np.zeros_like(gt_union)
        for class_id, mask in gt:
            gt_union[class_id] |= mask
        for class_id, _, mask in pred:
            pred_union[class_id] |= mask
        if not gt:
            background_total += 1
            background_clean += int(not pred)
        per_image_iou = []
        for class_id in range(len(CLASS_NAMES)):
            g, p = gt_union[class_id], pred_union[class_id]
            if not g.any() and not p.any():
                continue
            inter = float(np.logical_and(g, p).sum()); uni = float(np.logical_or(g, p).sum())
            intersection[class_id] += inter; union[class_id] += uni
            dice_num[class_id] += 2 * inter; dice_den[class_id] += float(g.sum() + p.sum())
            per_image_iou.append(inter / max(1.0, uni))
        rows.append({"sample_id": row.sample_id, "image_label": row.image_label,
                     "n_gt": len(gt), "n_pred": len(pred),
                     "mean_iou": round(float(np.mean(per_image_iou)), 4) if per_image_iou else None})
    valid = union > 0
    summary = {
        "conf": conf,
        "mIoU": round(float((intersection[valid] / union[valid]).mean()), 4) if valid.any() else None,
        "Dice": round(float((dice_num[valid] / np.maximum(1.0, dice_den[valid])).mean()), 4) if valid.any() else None,
        "per_class_IoU": {CLASS_NAMES[i]: round(float(intersection[i] / union[i]), 4)
                          for i in range(len(CLASS_NAMES)) if union[i] > 0},
        "background_images": background_total,
        "background_clean_rate": round(background_clean / max(1, background_total), 4),
    }
    return summary, pd.DataFrame(rows)


SEMANTIC, SEMANTIC_ROWS = semantic_scores(BEST_CONF, "test", EVAL_MAX_IMAGES)
SEMANTIC_ROWS.to_csv(ARTIFACTS_DIR / "test_semantic_scores.csv", index=False)
print(json.dumps(SEMANTIC, indent=2))


def cpu_latency(n_images: int = 30) -> dict:
    frame = EXPORT[EXPORT["split"] == "test"].head(n_images)
    model = YOLO(str(best_ckpt))
    paths = frame["image_path"].tolist()
    for path in paths[:3]:
        model.predict(path, imgsz=TRAIN_ARGS["imgsz"], device="cpu", verbose=False)
    timings = []
    for path in paths:
        started = time.perf_counter()
        model.predict(path, imgsz=TRAIN_ARGS["imgsz"], conf=BEST_CONF, device="cpu", verbose=False)
        timings.append((time.perf_counter() - started) * 1000.0)
    return {"n_images": len(timings), "imgsz": TRAIN_ARGS["imgsz"],
            "cpu_ms_mean": round(float(np.mean(timings)), 2),
            "cpu_ms_p95": round(float(np.percentile(timings, 95)), 2)}


LATENCY = cpu_latency(int(os.environ.get("LATENCY_IMAGES", "30")))
print(json.dumps(LATENCY, indent=2))

{
  "conf": 0.5,
  "mIoU": 0.6986,
  "Dice": 0.8155,
  "per_class_IoU": {
    "LeafMiner": 0.7129,
    "PowderyMildew": 0.8344,
    "Rust": 0.8423,
    "AlgalLeafSpot": 0.808,
    "BrownSpot": 0.6506,
    "Hispa": 0.5299,
    "LeafBlast": 0.5123
  },
  "background_images": 208,
  "background_clean_rate": 0.8029
}
{
  "n_images": 30,
  "imgsz": 1024,
  "cpu_ms_mean": 205.35,
  "cpu_ms_p95": 223.16
}


## 10. Artifacts

Everything needed to audit or reproduce this run: checkpoint, resolved dataset and training
configuration, environment, training curves, validation metrics, the confidence sweep, per-image
predictions, and a model card that states the operating point.

In [12]:
shutil.copy2(best_ckpt, ARTIFACTS_DIR / f"best_yolo26n_seg_{TARGET_DOMAIN}.pt")
shutil.copy2(best_ckpt, ARTIFACTS_DIR / "best.pt")
shutil.copy2(DATA_YAML, ARTIFACTS_DIR / "data.yaml")
args_yaml = Path(train_summary.get("run_dir", "")) / "args.yaml"
if args_yaml.exists():
    shutil.copy2(args_yaml, ARTIFACTS_DIR / "ultralytics_args.yaml")

RUN_MANIFEST = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "domain": TARGET_DOMAIN,
    "classes": CLASS_NAMES,
    "dataset": {
        "version": DATASET_VERSION,
        "root": str(DATASET_ROOT),
        "images_version": IMAGES_VERSION,
        "repair_config": REPAIR_CONFIG,
        "exported_counts": EXPORT.groupby("split")["num_instances"].agg(["size", "sum"]).to_dict(),
        "negative_train_ratio": NEGATIVE_TRAIN_RATIO,
    },
    "model": {"weights_init": TRAIN_ARGS["model"], "task": "instance_segmentation"},
    "train_args": TRAIN_ARGS,
    "training": train_summary,
    "operating_point": {"conf": BEST_CONF, "iou_nms": 0.7, "selected_on": CONF_SELECTION},
    "metrics": {
        "ultralytics_val": VALIDATION.get("val"),
        "ultralytics_test": VALIDATION.get("test"),
        "coco_test_original_resolution": COCO_METRICS,
        "semantic_test": SEMANTIC,
        "latency": LATENCY,
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "ultralytics": ultralytics.__version__,
    },
}
(ARTIFACTS_DIR / "run_manifest.json").write_text(json.dumps(RUN_MANIFEST, indent=2, default=str), encoding="utf-8")

summary_row = {
    "run_id": RUN_ID, "domain": TARGET_DOMAIN, "conf": BEST_CONF,
    "mask_mAP50_coco": COCO_METRICS.get("mask_mAP50"),
    "mask_mAP50_95_coco": COCO_METRICS.get("mask_mAP50_95"),
    "box_mAP50_coco": COCO_METRICS.get("box_mAP50"),
    "box_mAP50_95_coco": COCO_METRICS.get("box_mAP50_95"),
    "mask_mAP50_ultralytics": (VALIDATION.get("test") or {}).get("mask_mAP50"),
    "mIoU": SEMANTIC.get("mIoU"), "Dice": SEMANTIC.get("Dice"),
    "background_clean_rate": SEMANTIC.get("background_clean_rate"),
    "cpu_ms_mean": LATENCY.get("cpu_ms_mean"),
    "checkpoint_mb": round(Path(best_ckpt).stat().st_size / 1e6, 2),
}
SUMMARY = pd.DataFrame([summary_row])
SUMMARY.to_csv(ARTIFACTS_DIR / "summary.csv", index=False)
display(SUMMARY)

model_card = f"""# YOLO26-seg - {TARGET_DOMAIN.title()} leaf disease instance segmentation

Run `{RUN_ID}` | dataset `{DATASET_VERSION}` (repaired, leak-free grouped splits)

## Task
Instance segmentation of {TARGET_DOMAIN} leaf disease. Detection classes: {CLASS_NAMES}.
`Healthy` is an image-level label, not a class: a healthy leaf is expected to produce no instance.

## Operating point
conf = {BEST_CONF} (selected on the validation split by mask-F1), NMS IoU = 0.7,
imgsz = {TRAIN_ARGS['imgsz']}.

## Test metrics (COCO, original resolution)
- mask mAP@50: {COCO_METRICS.get('mask_mAP50')}
- mask mAP@50:95: {COCO_METRICS.get('mask_mAP50_95')}
- box mAP@50: {COCO_METRICS.get('box_mAP50')}
- box mAP@50:95: {COCO_METRICS.get('box_mAP50_95')}
- mIoU: {SEMANTIC.get('mIoU')} | Dice: {SEMANTIC.get('Dice')}
- background images returning nothing: {SEMANTIC.get('background_clean_rate')}
- CPU latency: {LATENCY.get('cpu_ms_mean')} ms/image (imgsz {TRAIN_ARGS['imgsz']})

## Known limits
- The model is closed-set. Out-of-domain images require the serving-side rejection gate;
  `background_clean_rate` only measures healthy leaves of the same domain.
- Rice labels come from two annotation protocols (studio whole-leaf vs field lesions) and the
  capture sessions correlate with classes; see `reports/` in the dataset version.
"""
(ARTIFACTS_DIR / "README.md").write_text(model_card, encoding="utf-8")
print(sorted(p.name for p in ARTIFACTS_DIR.iterdir()))


,run_id,domain,conf,mask_mAP50_coco,mask_mAP50_95_coco,box_mAP50_coco,box_mAP50_95_coco,mask_mAP50_ultralytics,mIoU,Dice,background_clean_rate,cpu_ms_mean,checkpoint_mb
0,20260922T115556Z,joint,0.5,0.6169,0.6106,0.6169,0.6011,0.775272,0.6986,0.8155,0.8029,205.35,6.61


['BoxPR_curve.png', 'MaskPR_curve.png', 'README.md', 'best.pt', 'best_yolo26n_seg_joint.pt', 'confusion_matrix_normalized.png', 'data.yaml', 'domain_breakdown_metrics.json', 'label_distribution.csv', 'label_qa', 'results.png', 'run_manifest.json', 'summary.csv', 'test_ground_truth.coco.json', 'test_ground_truth_coffee.coco.json', 'test_ground_truth_rice.coco.json', 'test_per_image_predictions.csv', 'test_predictions.coco.json', 'test_predictions_coffee.coco.json', 'test_predictions_rice.coco.json', 'test_semantic_scores.csv', 'training_curves.csv', 'ultralytics_args.yaml', 'val_confidence_sweep.csv']


## 11. Base ONNX Export (FP32)

Exports the full-precision **ONNX FP32** model (`yolo26_unified.onnx`) and the corresponding inference specification (`serving_contract.json`).

**Engineering Note:**
- Training and evaluation focus strictly on FP32 baseline metrics (mAP, mIoU, Dice).
- **Post-Training Quantization (PTQ INT8)** is conducted independently on local CPU to establish an empirical trade-off analysis (disk size, latency vs. accuracy drop).

In [13]:
if os.environ.get("EXPORT_ONNX", "1") == "1":
    exported = YOLO(str(best_ckpt)).export(format="onnx", imgsz=TRAIN_ARGS["imgsz"], opset=17,
                                           dynamic=False, simplify=True, nms=False)
    shutil.copy2(exported, ARTIFACTS_DIR / f"yolo26n_seg_{TARGET_DOMAIN}.onnx")
    if IS_JOINT:
        shutil.copy2(exported, ARTIFACTS_DIR / "yolo26_unified.onnx")
        # Base FP32 model ready for post-training quantization on CPU
    else:
        # Base FP32 model ready for post-training quantization on CPU
        pass
        
    (ARTIFACTS_DIR / "serving_contract.json").write_text(json.dumps({
        "input": {"name": "images", "shape": [1, 3, TRAIN_ARGS["imgsz"], TRAIN_ARGS["imgsz"]],
                  "preprocess": "letterbox to square, pad 114, RGB, /255"},
        "classes": CLASS_NAMES,
        "conf": BEST_CONF, "iou_nms": 0.7,
        "postprocess": "decode seg protos, NMS, then unletterbox to original resolution",
        "image_level_labels": REPAIR_CONFIG["class_policy"]["image_level_labels"],
        "target_domain": TARGET_DOMAIN,
        "precision": "FP32",
        "stage": "base_inference_model",
    }, indent=2), encoding="utf-8")
    print(f"Exported ONNX with serving contract for {len(CLASS_NAMES)} classes.")
else:
    print("EXPORT_ONNX=0, skipping. Serving must reuse the letterbox + segmentation decode above.")

# Create a zip archive of all artifacts for convenient 1-click download on Kaggle
zip_path = shutil.make_archive(str(ARTIFACTS_DIR), 'zip', ARTIFACTS_DIR)
print(f"\nAll artifacts zipped to: {zip_path} ({round(Path(zip_path).stat().st_size / 1e6, 2)} MB)")
print("Artifacts directory contents:")
for p in sorted(ARTIFACTS_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(ARTIFACTS_DIR)
        size_kb = round(p.stat().st_size / 1024, 1)
        print(f" - {str(rel):40s} : {size_kb:8.1f} KB")

Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n-seg summary (fused): 139 layers, 2,690,249 parameters, 0 gradients, 23.8 GFLOPs

PyTorch: starting from '/kaggle/working/runs/yolo26_seg/yolo26n_seg_joint_20260922T115556Z/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) ((1, 300, 38), (1, 32, 256, 256)) (6.3 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 555ms
 Downloaded onnxruntime
Prepared 2 packages in 527ms
Installed 2 packages in 18ms
 + onnxruntime==1.30.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 2.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.2